# Transformer Testing

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from modellens import ModelLens


# Load model
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11553.48it/s]


In [2]:
# Create lens
lens = ModelLens(model)
lens.adapter.set_tokenizer(tokenizer)

# Check summary
print(lens)
print(lens.summary())

ModelLens(
  backend=huggingface,
  architecture=transformer,
  params=124,439,808,
  analyses=['activation_patching', 'attention_maps', 'embeddings', 'hooks', 'layer_probing', 'residual_stream']
)
{'backend': 'huggingface', 'architecture': 'transformer', 'total_parameters': 124439808, 'layer_count': 163, 'hooks_attached': 0, 'activations_captured': [], 'available_analyses': ['activation_patching', 'attention_maps', 'embeddings', 'hooks', 'layer_probing', 'residual_stream'], 'unavailable_analyses': ['feature_map_analysis', 'filter_analysis', 'gate_analysis']}


In [3]:
# Tokenize input
inputs = tokenizer("The capital of France is", return_tensors="pt")

# Logit lens / layer probe
results = lens.layer_probe(inputs, top_k=5)
from modellens.analysis.logit_lens import decode_logit_lens
decoded = decode_logit_lens(results, tokenizer=tokenizer)
for layer, preds in list(decoded.items())[:3]:
    print(f"{layer}: {preds}")

transformer.wte: [(' is', 0.0035959226079285145), (' has', 0.0011945082806050777), (' was', 0.00097653764532879), (' does', 0.0005770876887254417), (' are', 0.0005422612302936614)]
transformer.wpe: [('theless', 8.032188634388149e-05), ('Thumbnail', 7.765968621242791e-05), ('lust', 7.400718459393829e-05), ('מ', 7.280485442606732e-05), (' gaping', 7.028273103060201e-05)]
transformer.drop: [(' is', 0.003390722908079624), (' has', 0.0012133164564147592), (' was', 0.000962700869422406), (' does', 0.0006336725200526416), (' will', 0.0006186149548739195)]


In [4]:
# Attention maps
attn = lens.attention_map(inputs)
print(f"Attention layers: {attn['num_layers']}")

Attention layers: 12


In [5]:
# Residual stream
residual = lens.residual_stream(inputs)
print(f"Layers analyzed: {residual['num_layers_analyzed']}")

Layers analyzed: 11


In [7]:
# Embeddings
emb = lens.embeddings(inputs)
print(f"Embedding dim: {emb['embed_dim']}")
print(f"Seq length: {emb['seq_length']}")
print(f"Similarity matrix shape: {emb['similarity_matrix'].shape}")

Embedding dim: 768
Seq length: 5
Similarity matrix shape: torch.Size([5, 5])


# CNN Testing

In [10]:
import torch
import torch.nn as nn
from modellens import ModelLens

# Simple CNN
class TestCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(32, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.flatten(1)
        return self.classifier(x)

cnn = TestCNN()
lens = ModelLens(cnn)

print(lens)
print(f"\nAvailable: {lens.available_analyses()}")

ModelLens(
  backend=pytorch,
  architecture=convolutional,
  params=5,130,
  analyses=['activation_patching', 'feature_map_analysis', 'filter_analysis', 'hooks', 'layer_probing']
)

Available: ['activation_patching', 'feature_map_analysis', 'filter_analysis', 'hooks', 'layer_probing']


In [11]:
# Test filter analysis
img = torch.randn(1, 1, 28, 28)
filters = lens.filter_analysis(img)
print(f"\nTotal filters: {filters['total_filters']}")
print(f"Dead filters: {filters['total_dead_filters']}")


Total filters: 48
Dead filters: 0


In [12]:
# Test feature map analysis
fmaps = lens.feature_maps(img)
print(f"\nLayers tracked: {fmaps['num_layers_tracked']}")
print(f"Spatial reduction: {fmaps['spatial_reduction']}x")
print(f"Channel expansion: {fmaps['channel_expansion']}x")


Layers tracked: 5
Spatial reduction: 28.0x
Channel expansion: 0.625x


In [13]:
for entry in fmaps['evolution']:
    print(f"{entry['layer']}: channels={entry['channels']}, spatial={entry['spatial_h']}x{entry['spatial_w']}")

conv1: channels=16, spatial=28x28
pool1: channels=16, spatial=14x14
conv2: channels=32, spatial=14x14
pool2: channels=32, spatial=1x1
classifier: channels=10, spatial=1x1
